# CSU salary schedule folding

Attach the 2024 salary schedule to CSU payroll, then append UC records unchanged.
Inputs and the combined Parquet export are in `golden-datasets`.
Run the notebook from top to bottom after changing inputs or settings.

`final_csu_df` retains all matching details. `final_csu_trimmed_df` is a reduced
CSU view only; the export uses the full `combined_df` for validation charts.


In [1]:
import pandas as pd

salary_schedule_df = pd.read_csv('../data/raw/SalaryRecsCSU.csv')
csu_data_df = pd.read_parquet('../data/processed/golden_csu_2013-2024.parquet')
uc_data_df = pd.read_parquet('../data/processed/golden_uc_2013-2024.parquet')

In [2]:
# Drop supplemental pay; leave the source file unchanged.
supplemental_pay = csu_data_df["role"].eq("supplemental_teaching_pay").fillna(False)
dropped_supplemental_rows = int(supplemental_pay.sum())
csu_data_df = csu_data_df.loc[~supplemental_pay].copy().reset_index(drop=True)
print(f"Dropped {dropped_supplemental_rows:,} supplemental teaching pay rows; "
      f"retained {len(csu_data_df):,} CSU rows.")


Dropped 48,384 supplemental teaching pay rows; retained 124,849 CSU rows.


### Matching rules

- Match `title_tc` to the schedule first; use `title_scrapped` for missing information.
- Keep payroll classifications and appointment-specific bounds. Never combine AY and 12-month ranges.
- Directory fallback preserves a known appointment term, otherwise assumes AY.
- Use explicit payroll ranks before directory ranks. Lecturer jobs default to A (maritime: II), regardless of extracted rank.
- Generic Instructional Faculty ranks may be inferred from the nearest other year for the same campus/name. Conflicting nearest ranks stay unresolved.
- Drop supplemental pay; retain other unmatched rows with a reason.

All payroll years use the 2024 schedule. This follows the classification-first approach
in `Models/DataPrep/csu_data_prep.ipynb`, with the inference policy below.


### Settings

These settings control matching defaults and summary filters.


In [3]:
import re
from functools import lru_cache

# One schedule cutoff for every payroll year.
SALARY_SCHEDULE_YEAR = 2024
SALARY_CUTOFF = pd.Timestamp(SALARY_SCHEDULE_YEAR, 12, 31)
SALARY_BOUND_COLUMNS = ("Min Salary", "Max Salary")

# Defaults: missing appointment term, ordinary lecturer range, unranked chair/emeritus.
FALLBACK_APPOINTMENT_TERM = "academic year"
LECTURER_DEFAULT_RANGE = "a"
UNRANKED_CHAIR_EMERITUS_RANK = "professor"
MAX_TITLE_LENGTH = 350

COMBINED_RANK_LABELS = {
    "a asst instructor": {"instructor"},
    "b asst professor": {"assistant professor"},
    "c assoc prof prof": {"associate professor", "professor"},
}
FINAL_RANGE_LABELS = {"a": "lecturer class A", "range a": "lecturer class A"}

# Resolve tied candidates only when their usable monthly ranges agree.
ASSIGN_EQUIVALENT_MONTHLY_RANGES = True

# Defensive guard; supplemental rows are already dropped before matching.
EXCLUDED_COMPARISON_ROLES = {"supplemental_teaching_pay"}
EXCLUDED_ROLE_REASON = "Role excluded from salary comparison"

# Options: "instructor", "unmatched", "infer_from_other_years".
INSTRUCTIONAL_FACULTY_TITLE_POLICY = "infer_from_other_years"
INFERABLE_RANKS = {"instructor", "assistant professor", "associate professor", "professor"}

SUMMARY_YEAR = 2024  # None includes all years.
MIN_PAY_FOR_COMPARISON = 0  # Compare only pay above this threshold.


In [4]:
# Text normalization and rank extraction follow the reference notebook.
RANK_PATTERN = re.compile(
    r"\b(?:(assistant|asst|assis|associate|associat|assoc|assc)\s+"
    r"(?:(?:clinical|research|visiting|adjunct)\s+)*)?"
    r"(professor|prof)\b"
    r"|\b(lecturer|lecturtrer|instructor)(?:\s+(iii|ii|iv|i|[a-d]))?\b"
)


def normalize_title(value):
    """Normalize scraped titles, including joined words such as CoordinatorAssociate."""
    if pd.isna(value):
        return ""
    separated = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(value))
    return re.sub(r"[^a-z0-9]+", " ", separated.casefold()).strip()


def normalize_class_titles(titles):
    """Normalize schedule classification names without removing hyphens."""
    return (
        titles.astype("string").str.strip().str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
        .replace("", pd.NA)
    )


def rank_from_title_match(match):
    """Translate one regex match into a distinct professor or lecturer rank."""
    qualifier, professor, role, level = match.groups()
    if professor:
        if qualifier in {"assistant", "asst", "assis"}:
            return "assistant professor"
        return "associate professor" if qualifier else "professor"
    role = "lecturer" if role == "lecturtrer" else role
    return role + (f" {level}" if level else "")


def is_generic_instructional_faculty(title):
    """True when the whole title is just the generic class name 'Instructional Faculty'."""
    return normalize_title(title) == "instructional faculty"


def extract_title_rank(title):
    """Return a single rank; leave long or conflicting titles unresolved."""
    text = normalize_title(title)
    if len(text) > MAX_TITLE_LENGTH:
        return pd.NA
    ranks = {rank_from_title_match(match) for match in RANK_PATTERN.finditer(text)}
    if not ranks:
        if re.search(r"\b(?:emerit(?:us|a|i|u)?|chair(?:person|man|woman)?)\b", text):
            return UNRANKED_CHAIR_EMERITUS_RANK
        if text == "instructional faculty" and INSTRUCTIONAL_FACULTY_TITLE_POLICY == "instructor":
            return "instructor"
    return next(iter(ranks)) if len(ranks) == 1 else pd.NA


def explicit_payroll_rank(title):
    """Only explicit rank words count as payroll evidence; chair alone is not a rank."""
    text = normalize_title(title)
    if len(text) > MAX_TITLE_LENGTH:
        return pd.NA
    ranks = {rank_from_title_match(m) for m in RANK_PATTERN.finditer(text)}
    # A generic lecturer classification does not specify its range.
    ranks.discard("lecturer")
    return next(iter(ranks)) if len(ranks) == 1 else pd.NA


def appointment_term(title):
    text = normalize_title(title)
    if re.search(r"\b12\s*(?:month|mo|m)\b", text):
        return "12 month"
    if re.search(r"\b(?:academic year|ay)\b", text):
        return "academic year"
    return None


In [5]:
def parse_salary_amounts(values):
    """Convert dollar strings to numeric amounts; invalid values remain missing."""
    return pd.to_numeric(
        values.astype("string").str.replace(r"[$,]", "", regex=True), errors="coerce"
    )


def filter_schedule_to_cutoff(raw_schedule, cutoff):
    """Keep the latest eligible version of each class/range without changing the input."""
    schedule = raw_schedule.copy()
    schedule["Effective Date"] = pd.to_datetime(
        schedule["Effective Date"], format="%m/%d/%Y", errors="coerce"
    )
    schedule = schedule.loc[schedule["Effective Date"].le(cutoff)].copy()
    latest_dates = schedule.groupby(["Class", "Range"], dropna=False)[
        "Effective Date"
    ].transform("max")
    # Remove exact duplicates; retain conflicting rows for ambiguity checks.
    return schedule.loc[schedule["Effective Date"].eq(latest_dates)].drop_duplicates().reset_index(drop=True)


def prepare_schedule_columns(schedule):
    """Add normalized matching keys and parse salary bounds."""
    schedule = schedule.copy()
    schedule["_salary_row_id"] = pd.array(range(len(schedule)), dtype="Int64")
    schedule["_class_key"] = normalize_class_titles(schedule["Class Title"])
    schedule["_range_key"] = schedule["Range Title"].map(normalize_title)
    for column in SALARY_BOUND_COLUMNS:
        schedule[column] = parse_salary_amounts(schedule[column])
    schedule["Class"] = schedule["Class"].astype("string")
    return schedule

In [6]:
def build_matching_schedule(raw_schedule, cutoff):
    """Keep appointment-specific rows and their own bounds, as in the reference."""
    return prepare_schedule_columns(filter_schedule_to_cutoff(raw_schedule, cutoff))


In [7]:
def lecturer_range_keys():
    return {LECTURER_DEFAULT_RANGE, "range " + LECTURER_DEFAULT_RANGE}


def describe_available_ranges(candidates):
    """Keep the available labels in diagnostics when no unique match can be made."""
    labels = candidates["Range Title"].fillna("(blank)").astype(str).str.strip()
    return " | ".join(labels.drop_duplicates())


def unmatched_result(reason, available_ranges=""):
    """Represent an unresolved match without inventing a salary range."""
    return {
        "_salary_row_id": None, "matched_rank": None, "match_method": "unmatched",
        "unmatched_reason": reason, "available_range_titles": available_ranges,
    }


def select_equivalent_monthly_default(rows):
    """Choose a stable default only when usable monthly candidates have identical ranges."""
    usable = rows.loc[
        rows["Salary Per"].eq("M")
        & rows["Annual Factor"].gt(0)
        & rows["Min Salary"].notna()
        & rows["Max Salary"].ge(rows["Min Salary"])
    ]
    comparison_columns = ["_range_key", "Salary Per", "Annual Factor", "Min Salary", "Max Salary"]
    if usable.empty or len(usable[comparison_columns].drop_duplicates()) != 1:
        return rows.iloc[:0]
    # Break equivalent-range ties by class code.
    return usable.sort_values(["Class", "Effective Date"], ascending=[True, False]).head(1)


def select_unique_range(rows, method, available_ranges):
    """Select one row or the documented equivalent-monthly default."""
    if len(rows) > 1 and ASSIGN_EQUIVALENT_MONTHLY_RANGES:
        default = select_equivalent_monthly_default(rows)
        if not default.empty:
            rows = default
            method += " / assumed equivalent monthly range; lowest class code"
    if len(rows) != 1:
        return unmatched_result("Multiple schedule rows fit the rule", available_ranges)
    row = rows.iloc[0]
    return {
        "_salary_row_id": int(row["_salary_row_id"]),
        "matched_rank": row["_range_key"] or None, "match_method": method,
        "unmatched_reason": "", "available_range_titles": available_ranges,
    }

In [8]:
def choose_classification(schedule, groups, payroll_key, scraped_key, scraped_rank):
    """Use exact payroll class first; directory evidence only fills a missing class."""
    if payroll_key in groups:
        return groups[payroll_key], "title_tc classification"
    if scraped_key in groups:
        candidates = groups[scraped_key]
        payroll_term = appointment_term(payroll_key)
        directory_term = appointment_term(scraped_key)
        if payroll_term and directory_term and payroll_term != directory_term:
            return schedule.iloc[:0], "conflicting appointment terms"
        return candidates, "title_scrapped classification fallback"
    rank = scraped_rank or ""
    if rank.startswith("lecturer"):
        classification = "lecturer"
    elif rank in INFERABLE_RANKS or is_generic_instructional_faculty(scraped_key):
        classification = "instructional faculty"
    else:
        return schedule.iloc[:0], "classification unavailable"
    term = appointment_term(payroll_key) or appointment_term(scraped_key)
    method = "title_scrapped classification fallback"
    if term is None:
        term = FALLBACK_APPOINTMENT_TERM
        method += " / assumed " + term
    target = classification + "-" + term
    return groups.get(target, schedule.iloc[:0]), method


def match_salary_range(schedule, groups, role, payroll_key, scraped_key, rank, scraped_rank):
    if role in EXCLUDED_COMPARISON_ROLES:
        return unmatched_result(EXCLUDED_ROLE_REASON)
    candidates, method = choose_classification(schedule, groups, payroll_key, scraped_key, scraped_rank)
    if candidates.empty:
        return unmatched_result("Job classification not found: " + method)
    available = describe_available_ranges(candidates)
    if len(candidates) == 1:
        return select_unique_range(candidates, method + " / single salary range", available)
    class_key = candidates["_class_key"].iloc[0]
    if re.search(r"\blecturer\b", class_key):
        maritime = bool(re.search(r"\b(?:maritime|cma)\b", class_key))
        preferred = {"lecturer ii"} if maritime else lecturer_range_keys()
        selected = candidates.loc[candidates["_range_key"].isin(preferred)]
        if selected.empty:
            return unmatched_result("Lecturer default not available for this job", available)
        default_label = "Lecturer II" if maritime else "Range " + LECTURER_DEFAULT_RANGE.upper()
        return select_unique_range(selected, method + " / default " + default_label, available)
    if rank is None:
        return unmatched_result("Job has multiple ranges; titles do not identify a rank", available)
    compatible = candidates["_range_key"].map(
        lambda label: rank in COMBINED_RANK_LABELS.get(label, {label})
    )
    if compatible.any():
        return select_unique_range(candidates.loc[compatible], method + " / compatible rank", available)
    return unmatched_result("Title rank unavailable under payroll job; job retained", available)


def build_salary_matcher(schedule):
    groups = {key: rows for key, rows in schedule.dropna(subset=["_class_key"]).groupby("_class_key")}

    @lru_cache(maxsize=None)
    def match(role, payroll_key, scraped_key, rank, scraped_rank):
        return match_salary_range(schedule, groups, role, payroll_key, scraped_key, rank, scraped_rank)
    return match


In [9]:
def infer_rank_from_other_years(csu):
    """Infer from the nearest other year; reject conflicting nearest ranks."""
    csu = csu.copy()
    if "rank_source" not in csu:
        csu["rank_source"] = pd.Series(pd.NA, index=csu.index, dtype="string")
        csu.loc[csu["extracted_rank"].notna(), "rank_source"] = "title"
    if INSTRUCTIONAL_FACULTY_TITLE_POLICY != "infer_from_other_years":
        return csu
    generic = csu["title_scrapped"].map(is_generic_instructional_faculty) & csu["extracted_rank"].isna()
    known = csu.loc[
        csu["extracted_rank"].isin(INFERABLE_RANKS),
        ["campus", "name", "year", "extracted_rank"],
    ].drop_duplicates()
    targets = csu.loc[generic, ["campus", "name", "year"]].reset_index().rename(columns={"index": "_row"})
    pairs = targets.merge(known, on=["campus", "name"], suffixes=("", "_known"))
    # Evidence must come from a different payroll year.
    pairs = pairs.loc[pairs["year"].ne(pairs["year_known"])].copy()
    pairs["_gap"] = (pairs["year"] - pairs["year_known"]).abs()
    nearest_gap = pairs.groupby("_row")["_gap"].transform("min")
    closest = pairs.loc[pairs["_gap"].eq(nearest_gap)]
    # Check every equally near year before choosing a source year.
    consistent = closest.groupby("_row")["extracted_rank"].transform("nunique").eq(1)
    nearest = (
        closest.loc[consistent].sort_values(["_row", "year_known"])
        .drop_duplicates("_row").set_index("_row")
    )
    csu.loc[nearest.index, "extracted_rank"] = nearest["extracted_rank"].array
    csu.loc[nearest.index, "rank_source"] = "inferred from " + nearest["year_known"].astype(str).array
    return csu


def add_title_matching_fields(csu_data):
    """Keep payroll classification/rank evidence ahead of directory fallback evidence."""
    csu = csu_data.copy()
    csu["_payroll_key"] = normalize_class_titles(csu["title_tc"])
    csu["_scraped_key"] = normalize_class_titles(csu["title_scrapped"])
    payroll_rank = csu["title_tc"].map(explicit_payroll_rank)
    csu["_scraped_rank"] = csu["title_scrapped"].map(extract_title_rank)
    csu["extracted_rank"] = payroll_rank.combine_first(csu["_scraped_rank"])
    csu["rank_source"] = pd.Series(pd.NA, index=csu.index, dtype="string")
    csu.loc[payroll_rank.notna(), "rank_source"] = "title_tc"
    csu.loc[payroll_rank.isna() & csu["_scraped_rank"].notna(), "rank_source"] = "title_scrapped"
    return infer_rank_from_other_years(csu)


def match_csu_rows(csu, matcher):
    """Return exactly one selection per input row."""
    fields = ["role", "_payroll_key", "_scraped_key", "extracted_rank", "_scraped_rank"]
    choices = []
    for row in csu[fields].itertuples(index=False, name=None):
        values = [None if pd.isna(value) else value for value in row]
        choices.append(matcher(*values))
    columns = ["_salary_row_id", "matched_rank", "match_method",
               "unmatched_reason", "available_range_titles"]
    selections = pd.DataFrame(choices, columns=columns)
    selections["_salary_row_id"] = pd.array(selections["_salary_row_id"], dtype="Int64")
    return selections


def join_salary_schedule(csu_data, schedule):
    """Attach the selected schedule row without duplicating or dropping personnel rows."""
    csu = add_title_matching_fields(csu_data)
    selections = match_csu_rows(csu, build_salary_matcher(schedule))
    for column in selections:
        csu[column] = selections[column].array
    public_schedule = schedule.drop(columns=["_class_key", "_range_key"])
    return csu.merge(
        public_schedule, on="_salary_row_id", how="left", validate="many_to_one",
        indicator="salary_match", suffixes=("", "_schedule"), sort=False,
    ).drop(columns=["_salary_row_id", "_payroll_key", "_scraped_key", "_scraped_rank"])


def apply_final_range_labels(joined_data):
    """Add the comparison year and apply display labels without changing source data."""
    final = joined_data.copy()
    final["salary_schedule_year"] = SALARY_SCHEDULE_YEAR
    range_keys = final["Range Title"].map(normalize_title)
    lecturer = final["Class Title"].astype("string").str.contains("lecturer", case=False, na=False)
    for key, label in FINAL_RANGE_LABELS.items():
        final.loc[lecturer & range_keys.eq(key), ["Range Title", "matched_rank"]] = label
    return final


def validate_final_csu_data(final, original, cutoff):
    """Check that original records survive unchanged and no future rates were joined."""
    assert len(final) == len(original)
    pd.testing.assert_frame_equal(
        final[original.columns].reset_index(drop=True), original.reset_index(drop=True)
    )
    matched_dates = final.loc[final["salary_match"].eq("both"), "Effective Date"]
    assert matched_dates.notna().all() and matched_dates.le(cutoff).all()

In [10]:
# Build and validate one schedule selection per CSU record.
schedule = build_matching_schedule(salary_schedule_df, SALARY_CUTOFF)
csu_salary_joined_df = join_salary_schedule(csu_data_df, schedule)
final_csu_df = apply_final_range_labels(csu_salary_joined_df)
validate_final_csu_data(final_csu_df, csu_data_df, SALARY_CUTOFF)
final_csu_df.head()

,year,campus,name,title_scrapped,role,title_tc,original_department,department,gross_pay,base_pay,...,Step 25,Step 26,Step 27,Step 28,Step 29,Step 30,Step 31,Step 32,salary_match,salary_schedule_year
0,2013,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",148840,97454,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
1,2014,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",164953,98207,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
2,2015,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",155361,100861,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
3,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",146317,102539,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024
4,2017,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",134508,99024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both,2024


### Salary comparisons

Compare positive annual `gross_pay` with monthly schedule bounds times `Annual Factor`.
`salary_expected` is the bounds' midpoint, not a prediction. All years use 2024 rates,
without FTE, partial-year, or inflation adjustments.

Percentages use comparable payroll records, not unique people. Missing pay, unmatched
ranges, and pay at/below the threshold are counted separately. Groups without comparable
records have missing percentages. Below-midpoint counts include below-minimum counts.


In [11]:
# Annual pay field used in summaries.
PAY_COMPARISON_COLUMN = "gross_pay"


def add_salary_comparison_columns(data):
    """Add the raw range midpoint and compatible annual comparison amounts."""
    result = data.copy()
    minimum = parse_salary_amounts(result["Min Salary"])
    maximum = parse_salary_amounts(result["Max Salary"])
    factor = pd.to_numeric(result["Annual Factor"], errors="coerce")
    valid_range = (result["salary_match"].eq("both") & minimum.ge(0)
                   & maximum.ge(minimum)).fillna(False)
    result["salary_range_mean"] = ((minimum + maximum) / 2).where(valid_range)
    # Annualize monthly rates only.
    monthly = result["Salary Per"].astype("string").str.strip().eq("M").fillna(False).astype(bool)
    annualizable = (valid_range & monthly & factor.gt(0)).fillna(False)
    result["salary_low"] = (minimum * factor).where(annualizable)
    result["salary_high"] = (maximum * factor).where(annualizable)
    result["salary_expected"] = (result["salary_low"] + result["salary_high"]) / 2
    return result


def filter_summary_year(data):
    """Restrict a table to SUMMARY_YEAR (no change when SUMMARY_YEAR is None)."""
    return data if SUMMARY_YEAR is None else data.loc[data["year"].eq(SUMMARY_YEAR)]


def summarize_salary_comparisons(data, group_column):
    """Summarize matched, unmatched, low-pay, and comparable records."""
    data = filter_summary_year(data)
    pay = parse_salary_amounts(data[PAY_COMPARISON_COLUMN])
    matched = data["salary_match"].eq("both")
    excluded = data["unmatched_reason"].eq(EXCLUDED_ROLE_REASON)
    pay_ok = pay.gt(MIN_PAY_FOR_COMPARISON).fillna(False)
    comparable = (matched & pay_ok & data["salary_low"].notna()
                  & data["salary_expected"].notna())
    low_pay = matched & pay.notna() & ~pay_ok
    flags = data[[group_column]].assign(
        excluded_role_records=excluded,
        unmatched_records=~matched & ~excluded,
        pay_at_or_below_threshold_records=low_pay,
        matched_not_comparable_records=matched & ~comparable & ~low_pay,
        comparable_records=comparable,
        below_range_records=(comparable & pay.lt(data["salary_low"])).fillna(False),
        below_expected_records=(comparable & pay.lt(data["salary_expected"])).fillna(False),
    )
    grouped = flags.groupby(group_column, dropna=False)
    count_columns = [column for column in flags.columns if column != group_column]
    summary = grouped[count_columns].sum()
    summary.insert(0, "total_records", grouped.size())
    denominator = summary["comparable_records"].replace(0, float("nan"))
    summary["below_range_percent"] = (100 * summary["below_range_records"] / denominator).round(2)
    summary["below_expected_percent"] = (100 * summary["below_expected_records"] / denominator).round(2)
    return summary


def summarize_record_counts(data, group_column):
    """Count all records in each group, including groups with a missing label."""
    summary = data.groupby(group_column, dropna=False).size().rename("records").to_frame()
    summary["percent_of_all_records"] = (100 * summary["records"] / len(data)).round(2)
    return summary


final_csu_df = add_salary_comparison_columns(final_csu_df)

### Reading the matching diagnostics

`match_method` combines a classification source with a range-selection rule (separated by `/`).

| Method component | Meaning |
| --- | --- |
| `title_tc classification` | Payroll title matched the schedule classification. |
| `title_scrapped classification fallback` | Directory title supplied the classification when payroll could not. |
| `assumed academic year` | Neither title specified an appointment term; AY was assumed. |
| `compatible rank` | Extracted/inferred rank selected a compatible range within the classification. |
| `single salary range` | Classification had one schedule row; no rank was needed. |
| `default Range A` | Ordinary lecturer classification used the configured default (A). |
| `default Lecturer II` | Maritime lecturer classification used its default range. |
| `assumed equivalent monthly range; lowest class code` | Equivalent usable monthly ranges were tied; lowest class code won. |
| `unmatched` | No schedule row selected; see `unmatched_reason`. |

| `rank_source` | Meaning |
| --- | --- |
| `title_tc` | Explicit rank extracted from the payroll title. |
| `title_scrapped` | Rank parsed from the directory title, including configured title assumptions. |
| `inferred from YYYY` | Rank borrowed from the same campus/name in the nearest other year; conflicting nearest ranks are rejected. |
| Missing / `No title rank extracted; single range or default` | No rank extracted; matched records used a single range or default. |
| `title` | Generic label only if the inference helper receives pre-extracted ranks without source labels. |

`rank_source` describes rank evidence, not necessarily the rule that selected the schedule:
single-range and lecturer-default matches can ignore that evidence. Counts below cover matched
records for methods/sources, and unmatched records for reasons; `AUDIT_YEAR` controls the year.


In [12]:
# Matching diagnostics across all years; set AUDIT_YEAR = 2024 to restrict them.
AUDIT_YEAR = None
audit = final_csu_df if AUDIT_YEAR is None else final_csu_df.loc[final_csu_df["year"].eq(AUDIT_YEAR)]
matched = audit["salary_match"].eq("both")

match_method_counts = audit.loc[matched, "match_method"].value_counts(dropna=False).rename("records").to_frame()
rank_source_counts = (
    audit.loc[matched, "rank_source"].fillna("No title rank extracted; single range or default")
    .value_counts().rename("records").to_frame()
)
# Show the actual source titles as well as the matching method/source.
matched_title_counts = (
    audit.loc[matched].groupby(["match_method", "rank_source", "title_tc", "title_scrapped"], dropna=False)
    .size().rename("records").sort_values(ascending=False).reset_index()
)
unmatched_reason_counts = (
    audit.loc[~matched, "unmatched_reason"].fillna("Reason missing").replace("", "Reason missing")
    .value_counts().rename("records").to_frame()
)
for label, table in [
    ("Classification matching method", match_method_counts),
    ("Rank evidence source (may be unused for single-range/default matches)", rank_source_counts),
    ("Actual titles on matched records", matched_title_counts),
    ("Unmatched reasons", unmatched_reason_counts),
]:
    print(label)
    display(table)
print(f"Matched: {matched.sum():,}; unmatched: {(~matched).sum():,}")


Classification matching method


,records
match_method,
title_tc classification / compatible rank,71784
title_tc classification / default Range A,39718
title_tc classification / single salary range,948
title_scrapped classification fallback / assumed academic year / compatible rank,114
title_scrapped classification fallback / compatible rank,58
title_scrapped classification fallback / assumed academic year / default Range A,11
title_scrapped classification fallback / default Range A,8


Rank evidence source (may be unused for single-range/default matches)


,records
rank_source,
title_scrapped,106891
No title rank extracted; single range or default,5453
inferred from 2023,37
inferred from 2014,33
inferred from 2021,31
inferred from 2020,30
inferred from 2022,30
inferred from 2017,28
inferred from 2018,27


Actual titles on matched records


,match_method,rank_source,title_tc,title_scrapped,records
0,title_tc classification / compatible rank,title_scrapped,instructional faculty - academic year,Professor,35793
1,title_tc classification / default Range A,title_scrapped,lecturer - academic year,Lecturer,23548
2,title_tc classification / compatible rank,title_scrapped,instructional faculty - academic year,Associate Professor,14106
3,title_tc classification / compatible rank,title_scrapped,instructional faculty - academic year,Assistant Professor,4863
4,title_tc classification / default Range A,<NA>,lecturer - academic year,NaN,2517
...,...,...,...,...,...
854,title_tc classification / single salary range,<NA>,coach - 12 month,"Head Coach, Women's Soccer",1
855,title_tc classification / single salary range,<NA>,coach - 12 month,"Head Coach, Women's Volleyball",1
856,title_tc classification / single salary range,<NA>,"student services professional, academic-relate...","Counseling Psychologist, Student Development",1
857,title_scrapped classification fallback / assum...,title_scrapped,research fellow - ne,Professor,1


Unmatched reasons


,records
unmatched_reason,
Job has multiple ranges; titles do not identify a rank,11189
Title rank unavailable under payroll job; job retained,989
Job classification not found: classification unavailable,30


Matched: 112,641; unmatched: 12,208


In [13]:
# Reduced view only; full data is still exported. Bounds retain schedule units.
schedule_columns_to_keep = [
    "matched_rank", "Min Salary", "Max Salary", "Class", "Class Title", "unmatched_reason",
]
columns_to_keep = list(dict.fromkeys([*csu_data_df.columns, *schedule_columns_to_keep]))
final_csu_trimmed_df = final_csu_df.loc[:, columns_to_keep].copy()
assert len(final_csu_trimmed_df) == len(final_csu_df)
display(final_csu_trimmed_df.head())


,year,campus,name,title_scrapped,role,title_tc,original_department,department,gross_pay,base_pay,overtime_pay,other_pay,matched_rank,Min Salary,Max Salary,Class,Class Title,unmatched_reason
0,2013,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",148840,97454,0,51386,professor,8593.0,15211.0,2360,INSTRUCTIONAL FACULTY - ACADEMIC YEAR,
1,2014,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",164953,98207,0,66746,professor,8593.0,15211.0,2360,INSTRUCTIONAL FACULTY - ACADEMIC YEAR,
2,2015,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",155361,100861,0,54500,professor,8593.0,15211.0,2360,INSTRUCTIONAL FACULTY - ACADEMIC YEAR,
3,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",146317,102539,0,43778,professor,8593.0,15211.0,2360,INSTRUCTIONAL FACULTY - ACADEMIC YEAR,
4,2017,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",134508,99024,0,35484,professor,8593.0,15211.0,2360,INSTRUCTIONAL FACULTY - ACADEMIC YEAR,


In [14]:
# Department buckets are stored in department; campus is the school identifier.
department_salary_summary = summarize_salary_comparisons(final_csu_df, "department")
department_salary_summary

,total_records,excluded_role_records,unmatched_records,pay_at_or_below_threshold_records,matched_not_comparable_records,comparable_records,below_range_records,below_expected_records,below_range_percent,below_expected_percent
department,,,,,,,,,,
"Agriculture, Food & Animal",431,0,17,59,0,355,132,277,37.18,78.03
Architecture & Environmental Design,365,0,26,41,0,298,136,256,45.64,85.91
"Business, Economics & Finance",1468,0,79,183,0,1206,275,445,22.8,36.9
Computer Science,777,0,120,88,0,569,167,287,29.35,50.44
Education & Public Administration,1227,0,84,174,0,969,423,749,43.65,77.3
Engineering,905,0,220,134,0,551,143,332,25.95,60.25
Humanities & Languages,2851,0,189,372,0,2290,827,1868,36.11,81.57
Medicine & Health Sciences,974,0,41,101,0,832,339,640,40.75,76.92
Other,790,0,272,92,0,426,259,322,60.8,75.59


In [15]:
school_salary_summary = summarize_salary_comparisons(final_csu_df, "campus")
school_salary_summary

,total_records,excluded_role_records,unmatched_records,pay_at_or_below_threshold_records,matched_not_comparable_records,comparable_records,below_range_records,below_expected_records,below_range_percent,below_expected_percent
campus,,,,,,,,,,
CSU Bakersfield,321,0,0,13,0,308,99,197,32.14,63.96
CSU Channel Islands,345,0,19,25,0,301,124,196,41.2,65.12
CSU Long Beach,1072,0,135,131,0,806,382,629,47.39,78.04
CSU Los Angeles,407,0,315,12,0,80,43,53,53.75,66.25
CSU Northridge,729,0,0,11,0,718,175,554,24.37,77.16
CSU San Bernardino,966,0,27,84,0,855,463,683,54.15,79.88
Cal Poly Humboldt,483,0,22,84,0,377,155,319,41.11,84.62
Cal Poly Pomona,653,0,7,30,0,616,220,430,35.71,69.81
Cal Poly San Luis Obispo,1328,0,227,321,0,780,157,539,20.13,69.1


In [16]:
records_by_year = summarize_record_counts(final_csu_df, "year")
records_by_year

,records,percent_of_all_records
year,,
2013,6426,5.15
2014,7054,5.65
2015,7757,6.21
2016,9423,7.55
2017,8862,7.10
2018,10374,8.31
2019,11036,8.84
2020,11428,9.15
2021,11879,9.51


In [17]:
records_by_school = summarize_record_counts(filter_summary_year(final_csu_df), "campus")
records_by_school

,records,percent_of_all_records
campus,,
CSU Bakersfield,321,2.26
CSU Channel Islands,345,2.43
CSU Long Beach,1072,7.54
CSU Los Angeles,407,2.86
CSU Northridge,729,5.13
CSU San Bernardino,966,6.80
Cal Poly Humboldt,483,3.40
Cal Poly Pomona,653,4.59
Cal Poly San Luis Obispo,1328,9.34


### Person-year-department totals

Sum remaining payroll lines by year, campus, name, and department, excluding supplemental pay.
People in multiple departments remain separate. Without employee IDs, identical names within
a group may combine different people. These totals do not change `final_csu_df`.


In [18]:
PERSON_YEAR_KEYS = ["year", "campus", "name", "department"]
PAY_COLUMNS = ["gross_pay", "base_pay", "overtime_pay", "other_pay"]

pay_aggregations = {"total_" + column: (column, "sum") for column in PAY_COLUMNS}

person_year_df = (
    csu_data_df.groupby(PERSON_YEAR_KEYS, dropna=False, as_index=False)
    .agg(**pay_aggregations, payroll_lines=("gross_pay", "size"))
)

# Which roles/titles were combined into each person-year (for inspection).
for column in ["role", "title_scrapped", "title_tc"]:
    distinct = csu_data_df.dropna(subset=[column]).drop_duplicates(PERSON_YEAR_KEYS + [column])
    joined = (distinct.sort_values(column).groupby(PERSON_YEAR_KEYS, dropna=False)[column]
              .agg(" | ".join).rename(column + "s").reset_index())
    person_year_df = person_year_df.merge(joined, on=PERSON_YEAR_KEYS, how="left", validate="one_to_one")

assert person_year_df["total_gross_pay"].sum() == csu_data_df["gross_pay"].sum()  # nothing lost
assert person_year_df["payroll_lines"].sum() == len(csu_data_df)
person_year_df.head()

,year,campus,name,department,total_gross_pay,total_base_pay,total_overtime_pay,total_other_pay,payroll_lines,roles,title_scrappeds,title_tcs
0,2013,CSU Bakersfield,Abbas Grammy,"Business, Economics & Finance",148840,97454,0,51386,1,ladder_faculty,Professor Emeritus,instructional faculty - academic year
1,2013,CSU Bakersfield,Abigael Flachmann,Humanities & Languages,38033,33615,0,4418,1,lecturer,Lecturer in English,lecturer - academic year
2,2013,CSU Bakersfield,Alemseghed Kebede,Social Sciences & Communication,77329,73680,0,3648,1,ladder_faculty,Professor,instructional faculty - academic year
3,2013,CSU Bakersfield,Alexander Dzyubenko,Engineering,82383,82383,0,0,1,ladder_faculty,Professor Emeritus,instructional faculty - academic year
4,2013,CSU Bakersfield,Allison Fluckey,Social Sciences & Communication,57629,57105,0,524,1,lecturer,Lecturer in Psychology,lecturer - academic year


In [19]:
person_year_summary_data = filter_summary_year(person_year_df)
active = person_year_summary_data.loc[person_year_summary_data["total_gross_pay"].gt(MIN_PAY_FOR_COMPARISON)]

overall_person_year_summary = pd.Series({
    "person_year_department_groups": len(person_year_summary_data),
    "total_pay_at_or_below_threshold": int((~person_year_summary_data["total_gross_pay"].gt(MIN_PAY_FOR_COMPARISON)).sum()),
    "multi_line_groups": int(person_year_summary_data["payroll_lines"].gt(1).sum()),
    "mean_total_pay": active["total_gross_pay"].mean(),
    "median_total_pay": active["total_gross_pay"].median(),
    "p25_total_pay": active["total_gross_pay"].quantile(0.25),
    "p75_total_pay": active["total_gross_pay"].quantile(0.75),
}).round(0)

person_year_summary_by_campus = active.groupby("campus").agg(
    person_year_department_groups=("total_gross_pay", "size"),
    multi_line_groups=("payroll_lines", lambda s: int((s > 1).sum())),
    mean_total_pay=("total_gross_pay", "mean"),
    median_total_pay=("total_gross_pay", "median"),
).round(0)

display(overall_person_year_summary)
person_year_summary_by_campus

person_year_department_groups       12057.0
total_pay_at_or_below_threshold       399.0
multi_line_groups                    1799.0
mean_total_pay                     103649.0
median_total_pay                   106862.0
p25_total_pay                       63301.0
p75_total_pay                      136166.0
dtype: float64

,person_year_department_groups,multi_line_groups,mean_total_pay,median_total_pay
campus,,,,
CSU Bakersfield,294,23,113135.0,106828.0
CSU Channel Islands,308,15,92554.0,91644.0
CSU Long Beach,856,138,90899.0,82648.0
CSU Los Angeles,363,33,110410.0,120174.0
CSU Northridge,709,17,121511.0,123669.0
CSU San Bernardino,829,74,77960.0,69885.0
Cal Poly Humboldt,361,75,90062.0,88372.0
Cal Poly Pomona,608,18,98450.0,96048.0
Cal Poly San Luis Obispo,781,395,137926.0,125421.0


In [20]:
# Append UC unchanged; CSU-only schedule fields are missing for UC.
SOURCE_COLUMN = "source_system"

csu_part = final_csu_df.copy()
uc_part = uc_data_df.copy()
assert SOURCE_COLUMN not in csu_part.columns and SOURCE_COLUMN not in uc_part.columns

csu_part[SOURCE_COLUMN] = "CSU"
uc_part[SOURCE_COLUMN] = "UC"

# merge indicator is categorical; object avoids category-vs-missing-column dtype surprises
if "salary_match" in csu_part.columns:
    csu_part["salary_match"] = csu_part["salary_match"].astype("object")

# Order: original payroll fields, source, then schedule fields.
payroll_cols = list(dict.fromkeys([*csu_data_df.columns, *uc_data_df.columns]))
schedule_cols = [c for c in csu_part.columns if c not in payroll_cols and c != SOURCE_COLUMN]
ordered_cols = payroll_cols + [SOURCE_COLUMN] + schedule_cols

combined_df = pd.concat(
    [csu_part, uc_part], axis=0, join="outer", ignore_index=True, sort=False,
).reindex(columns=ordered_cols)

# Check source data survives unchanged.
for source, original in [("CSU", csu_data_df), ("UC", uc_data_df)]:
    pd.testing.assert_frame_equal(
        combined_df.loc[combined_df[SOURCE_COLUMN].eq(source), original.columns].reset_index(drop=True),
        original.reset_index(drop=True), check_dtype=False,
    )

# Check combined rows and columns.
assert set(combined_df.columns) == set(csu_part.columns) | set(uc_part.columns)
assert len(combined_df) == len(csu_part) + len(uc_part)
assert combined_df[SOURCE_COLUMN].value_counts().to_dict() == {"UC": len(uc_part), "CSU": len(csu_part)}

combined_df.groupby(SOURCE_COLUMN).size()

source_system
CSU    124849
UC     399975
dtype: int64

In [21]:
combined_df.head()

,year,campus,name,title_scrapped,role,title_tc,original_department,department,gross_pay,base_pay,...,Step 29,Step 30,Step 31,Step 32,salary_match,salary_schedule_year,salary_range_mean,salary_low,salary_high,salary_expected
0,2013,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",148840,97454,...,NaN,NaN,NaN,NaN,both,2024.0,11902.0,103116.0,182532.0,142824.0
1,2014,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",164953,98207,...,NaN,NaN,NaN,NaN,both,2024.0,11902.0,103116.0,182532.0,142824.0
2,2015,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",155361,100861,...,NaN,NaN,NaN,NaN,both,2024.0,11902.0,103116.0,182532.0,142824.0
3,2016,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",146317,102539,...,NaN,NaN,NaN,NaN,both,2024.0,11902.0,103116.0,182532.0,142824.0
4,2017,CSU Bakersfield,Abbas Grammy,Professor Emeritus,ladder_faculty,instructional faculty - academic year,Economics,"Business, Economics & Finance",134508,99024,...,NaN,NaN,NaN,NaN,both,2024.0,11902.0,103116.0,182532.0,142824.0


In [22]:
# Check record counts, audit totals, labels, and inferred years before export.
assert not final_csu_df["role"].eq("supplemental_teaching_pay").any()
assert match_method_counts["records"].sum() + unmatched_reason_counts["records"].sum() == len(audit)

summary = summarize_salary_comparisons(final_csu_df, "campus")
status_columns = [
    "excluded_role_records", "unmatched_records", "pay_at_or_below_threshold_records",
    "matched_not_comparable_records", "comparable_records",
]
assert summary[status_columns].sum(axis=1).eq(summary["total_records"]).all()
assert summary["below_range_records"].le(summary["below_expected_records"]).all()
assert summary["below_expected_records"].le(summary["comparable_records"]).all()

inferred = final_csu_df["rank_source"].fillna("").str.startswith("inferred from ")
source_year = final_csu_df.loc[inferred, "rank_source"].str.removeprefix("inferred from ").astype(int)
assert source_year.ne(final_csu_df.loc[inferred, "year"]).all()

lecturer_label = final_csu_df["matched_rank"].eq("lecturer class A")
assert final_csu_df.loc[lecturer_label, "Class Title"].str.contains("lecturer", case=False).all()
assert list(final_csu_trimmed_df.columns) == columns_to_keep
print("Validation passed: records, summaries, inference years, labels, and trimmed columns.")


Validation passed: records, summaries, inference years, labels, and trimmed columns.


In [23]:
from pathlib import Path

output_path = Path("../data/processed/csu_uc_salary_schedule_2013-2024.parquet")
combined_df.to_parquet(output_path, index=False)
print(f"Saved {len(combined_df):,} records to {output_path.resolve()}")

Saved 524,824 records to C:\Users\ginny\Classes-Other\DataEng\CSC4693\data\processed\csu_uc_salary_schedule_2013-2024.parquet


In [24]:
salary_folded = pd.read_parquet('../data/processed/csu_uc_salary_schedule_2013-2024.parquet')

person_records = salary_folded.loc[
    salary_folded["name"].eq("Phuong Nguyen")
]

person_records.head(50)

,year,campus,name,title_scrapped,role,title_tc,original_department,department,gross_pay,base_pay,...,Step 29,Step 30,Step 31,Step 32,salary_match,salary_schedule_year,salary_range_mean,salary_low,salary_high,salary_expected
87962,2013,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - academic year,Humanities & Communication,Humanities & Languages,81764,73360,...,NaN,NaN,NaN,NaN,both,2024.0,6092.0,66084.0,80124.0,73104.0
87963,2014,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - academic year,Humanities & Communication,Humanities & Languages,80193,73920,...,NaN,NaN,NaN,NaN,both,2024.0,6092.0,66084.0,80124.0,73104.0
87964,2015,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - academic year,Humanities & Communication,Humanities & Languages,86311,79647,...,NaN,NaN,NaN,NaN,both,2024.0,6092.0,66084.0,80124.0,73104.0
87965,2016,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - 12 month,Humanities & Communication,Humanities & Languages,0,0,...,NaN,NaN,NaN,NaN,both,2024.0,6973.0,75456.0,91896.0,83676.0
87966,2016,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - academic year,Humanities & Communication,Humanities & Languages,96687,85195,...,NaN,NaN,NaN,NaN,both,2024.0,6092.0,66084.0,80124.0,73104.0
87967,2017,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,ladder_faculty,instructional faculty - academic year,Humanities & Communication,Humanities & Languages,17052,17356,...,NaN,NaN,NaN,NaN,both,2024.0,10674.0,81900.0,174276.0,128088.0
87968,2017,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - academic year,Humanities & Communication,Humanities & Languages,96559,85579,...,NaN,NaN,NaN,NaN,both,2024.0,6092.0,66084.0,80124.0,73104.0
87969,2018,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,ladder_faculty,instructional faculty - academic year,Humanities & Communication,Humanities & Languages,78034,70900,...,NaN,NaN,NaN,NaN,both,2024.0,10674.0,81900.0,174276.0,128088.0
87970,2018,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,lecturer,lecturer - academic year,Humanities & Communication,Humanities & Languages,100763,89098,...,NaN,NaN,NaN,NaN,both,2024.0,6092.0,66084.0,80124.0,73104.0
87971,2019,"California State University, Monterey Bay",Phuong Nguyen,Associate Professor,ladder_faculty,instructional faculty - academic year,Humanities & Communication,Humanities & Languages,101277,70113,...,NaN,NaN,NaN,NaN,both,2024.0,10674.0,81900.0,174276.0,128088.0
